# Exercise 2: Multi-Turn Conversation

Demonstrates how `previous_response_id` lets you chain conversation turns  
without manually managing a messages array. The API handles context accumulation
server-side -- you just pass the ID of the previous response.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

## Turn 1: Set the context

In [2]:
r1 = client.responses.create(
    model="gpt-4.1-mini",
    input=(
        "I'm evaluating three LLM providers for our enterprise platform: "
        "OpenAI, Anthropic, and Google. We need strong function calling, "
        "structured outputs, and enterprise security. What should I consider?"
    ),
)

print("User: I'm evaluating three LLM providers...\n")
print(f"Assistant: {r1.output_text}\n")
print(f"Response ID: {r1.id}")
print(f"Tokens: {r1.usage.input_tokens} in, {r1.usage.output_tokens} out")

User: I'm evaluating three LLM providers...

Assistant: Great question! When evaluating large language model (LLM) providers like **OpenAI, Anthropic, and Google** for your enterprise platform—specifically with priorities on **strong function calling, structured outputs, and enterprise security**—here are key considerations and how each provider generally stacks up:

---

### 1. **Function Calling**

- **OpenAI:**
  - Offers **built-in function calling APIs**, especially with GPT-4-turbo and GPT-4 models.
  - Supports declarative function calling where you define JSON schema for functions, and the model generates structured calls automatically.
  - This is widely regarded as one of the most mature and developer-friendly experiences for function calling.
  
- **Anthropic (Claude):**
  - Anthropic has capabilities for structured outputs and some level of guided outputs via prompt engineering.
  - As of now, function calling is less formally supported compared to OpenAI; more reliant on p

## Turn 2: Follow up using `previous_response_id`

The model remembers Turn 1 -- no message array needed.

In [3]:
r2 = client.responses.create(
    model="gpt-4.1-mini",
    input="Narrow it down. Which one has the best function calling reliability and structured output support?",
    previous_response_id=r1.id,
)

print("User: Which one has the best function calling reliability?\n")
print(f"Assistant: {r2.output_text}\n")
print(f"Response ID: {r2.id}")
print(f"Previous ID: {r1.id}")
print(f"Tokens: {r2.usage.input_tokens} in, {r2.usage.output_tokens} out")

User: Which one has the best function calling reliability?

Assistant: Narrowing it down specifically for **function calling reliability and structured output support**:

### **Clear winner: OpenAI**

- **Function calling:** OpenAI’s function calling API is **native, stable, and well-documented**, allowing you to define functions with JSON schemas and have GPT models generate precise, validated calls automatically.
- **Structured output:** OpenAI's support for strict JSON schema output enforcement makes structured data extraction reliable and easy to integrate.
- **Ecosystem:** Strong SDK support and production-ready tooling further add to the reliability.

### Anthropic and Google:

- **Anthropic:** More reliant on prompt engineering for structured outputs; function calling is not natively supported with schema enforcement.
- **Google:** Promising capabilities but still maturing in the function calling area; structured output often requires additional validation layers.

---

**Bottom

## Turn 3: Go deeper -- chain continues

In [4]:
r3 = client.responses.create(
    model="gpt-4.1-mini",
    input=(
        "OK, what about data residency and compliance? Our legal team needs "
        "SOC 2 Type II, HIPAA BAA, and EU data residency."
    ),
    previous_response_id=r2.id,
)

print("User: What about data residency and compliance?\n")
print(f"Assistant: {r3.output_text}\n")
print(f"Response ID: {r3.id}")
print(f"Previous ID: {r2.id}")
print(f"Tokens: {r3.usage.input_tokens} in, {r3.usage.output_tokens} out")

User: What about data residency and compliance?

Assistant: For **data residency and compliance** specifically with requirements for **SOC 2 Type II, HIPAA BAA, and EU data residency**, here’s how the three compare:

---

### **OpenAI**

- **SOC 2 Type II:** Available. OpenAI holds SOC 2 Type II compliance for their services.
- **HIPAA BAA:** Yes. OpenAI offers Business Associate Agreements (BAA) to support HIPAA compliance. This is crucial for handling protected health information.
- **EU Data Residency:**  
  - OpenAI offers options for EU data residency through partnerships and regional data hosting.  
  - Microsoft Azure OpenAI Service provides dedicated European region deployments, which can be an alternative if strict EU data residency is required.  
  - Recently, OpenAI has been expanding their data residency options—you’d want to confirm current offerings and contracts for your region.

---

### **Anthropic**

- **SOC 2 Type II:** Yes. Anthropic provides SOC 2 Type II compliant

## Token growth across turns

Input tokens grow each turn because the full conversation history is included  
server-side -- but you never had to manage that yourself.

In [ ]:
print(f"{'Turn':<6} {'Input':>8} {'Output':>8} {'Total':>8}")
print("-" * 34)
for i, r in enumerate([r1, r2, r3], 1):
    print(f"  {i:<4} {r.usage.input_tokens:>8} {r.usage.output_tokens:>8} {r.usage.total_tokens:>8}")

print("\nNotice: input tokens grow each turn because the full conversation is included.")
print("But YOU never had to manage a messages array -- the API does it via previous_response_id.")

## Interview one-liner

> **`previous_response_id`** gives you stateful multi-turn conversations without client-side message management -- the API accumulates context server-side, so your code stays simple while token usage grows predictably with conversation length.